# 🚀 StockGen AI - Unified Pipeline Test (Colab) [CORRECTED ORDER]
สมุดงานนี้จำลองการทำงานของ RunPod Worker แบบครบวงจร (เน้นคุณภาพสูงสุด)

**Flow การทำงาน (Pipeline ที่ถูกต้องสำหรับงาน Stock):**
1. รับภาพต้นฉบับ
2. **อัปสเกล (RealESRGAN)** -> ขยายภาพ 4 เท่า ให้รายละเอียดขอบเนียนกริบก่อน
3. **ลบพื้นหลัง (Rembg)** -> นำภาพ 4K ที่คมกริบมาลบฉากหลัง (ลดปัญหาขอบหลอน)
4. **ฝัง 300 DPI (piexif)** -> เตรียมพร้อมส่งขาย (PNG)

---

### 1. เช็ค GPU และติดตั้งไลบรารีทั้งหมด

In [ ]:
!nvidia-smi

print("\n📦 กำลังติดตั้งไลบรารี...")
!pip install rembg[gpu] realesrgan basicsr facexlib gfpgan piexif opencv-python-headless requests

### 2. นำเข้าโมดูลและโหลดรูปภาพทดสอบ

In [ ]:
import cv2
import numpy as np
import time
import requests
from PIL import Image
from io import BytesIO
import piexif

# โหลดภาพทดสอบ
image_url = 'https://raw.githubusercontent.com/danielgatis/rembg/master/examples/animal-1.jpg'
response = requests.get(image_url)
img_pil = Image.open(BytesIO(response.content)).convert("RGB")

print("📥 ภาพต้นฉบับโหลดเสร็จแล้ว ขนาด:", img_pil.size)
display(img_pil.resize((300, int(300 * img_pil.height / img_pil.width))))

### 3. โหลด AI Models (จำลองตอนเปิด Worker)
เราจะโหลดโมเดลทั้งหมดลง VRAM (RAM การ์ดจอ) รอไว้เลย

In [ ]:
print("🔄 1. กำลังเตรียม RealESRGAN (x4plus)...")
import sys
# ---------------------------------------------------------------------------
# Patch basicsr compatibility with newer PyTorch / torchvision
try:
    import torchvision.transforms.functional_tensor
except ImportError:
    try:
        import torchvision.transforms.functional as functional_tensor
        sys.modules['torchvision.transforms.functional_tensor'] = functional_tensor
    except ImportError:
        pass
# ---------------------------------------------------------------------------

from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

# ตั้งค่าโมเดลอัปสเกล
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    dni_weight=None,
    model=model,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=True, # ใช้ FP16
    gpu_id=0
)

print("🔄 2. กำลังเตรียม Rembg (u2net)...")
from rembg import remove, new_session
rembg_session = new_session('u2net')

print("✅ โหลดโมเดลทั้งหมดพร้อมใช้งานแล้ว!")

### 4. 🚀 RUN CORRECTED UNIFIED PIPELINE
ลำดับที่ถูกต้อง: ภาพต้นฉบับ -> อัปสเกล (RealESRGAN) 4x -> ลบพื้นหลัง (Rembg) -> ฝัง DPI 300

In [ ]:
print("\n=== 🚀 เริ่มต้น Unified Pipeline (SMART PIPELINE) ===")
total_start_time = time.time()

### STEP 1: ลบพื้นหลังบนภาพต้นฉบับ (ขนาดเล็ก)
step_start = time.time()
# ลบพื้นหลังบนภาพเล็ก จะเร็วมาก (ประมาณ 1 วิ) และเปิด Alpha Matting ได้เต็มที่
small_nobg_pil = remove(
    img_pil, 
    session=rembg_session, 
    alpha_matting=True,
    alpha_matting_foreground_threshold=240,
    alpha_matting_background_threshold=10,
    alpha_matting_erode_size=10
)
# ดึงแผ่นโปร่งใส (Alpha Mask) ออกมาเป็น Numpy Array
small_alpha_np = np.array(small_nobg_pil)[:, :, 3]
print(f"[1] ลบพื้นหลัง (ภาพต้นฉบับ + Alpha Matting) สำเร็จ ใช้เวลา: {time.time() - step_start:.2f} วินาที")

### STEP 2: อัปสเกลภาพต้นฉบับ (RealESRGAN)
step_start = time.time()
img_cv2 = np.array(img_pil)[:, :, ::-1]
upscaled_cv2, _ = upsampler.enhance(img_cv2, outscale=4)
print(f"[2] อัปสเกลภาพ 4x สำเร็จ ใช้เวลา: {time.time() - step_start:.2f} วินาที")

### STEP 3: ขยายขนาดแผ่น Alpha Mask และรวมร่าง
step_start = time.time()
out_h, out_w = upscaled_cv2.shape[:2]
# ใช้คณิตศาสตร์ (Lanczos4) ขยายขนาดแผ่นใสให้เท่ากับภาพ 4K (ใช้เวลาเสี้ยววินาที)
upscaled_alpha_np = cv2.resize(small_alpha_np, (out_w, out_h), interpolation=cv2.INTER_LANCZOS4)

# รวมร่าง (Merge) ภาพ 4K กับแผ่นใส 4K เข้าด้วยกัน
final_rgba_np = np.zeros((out_h, out_w, 4), dtype=np.uint8)
final_rgba_np[:, :, :3] = upscaled_cv2[:, :, ::-1] # แปลง BGR คืนเป็น RGB
final_rgba_np[:, :, 3] = upscaled_alpha_np         # ใส่แผ่นใสเข้าไป

final_nobg_pil = Image.fromarray(final_rgba_np, 'RGBA')
print(f"[3] ขยายแผ่น Alpha และรวมร่าง 4K สำเร็จ ใช้เวลา: {time.time() - step_start:.2f} วินาที")

### STEP 4: บันทึกและฝัง DPI
step_start = time.time()
buffer = BytesIO()
final_nobg_pil.save(buffer, format="PNG", dpi=(300, 300))
final_bytes = buffer.getvalue()
print(f"[4] ฝัง DPI 300 (PNG) สำเร็จ ใช้เวลา: {time.time() - step_start:.2f} วินาที")

total_time = time.time() - total_start_time
print(f"\n🎉 เสร็จสิ้นกระบวนการทั้งหมด! เวลาสุทธิ: {total_time:.2f} วินาที")
print(f"📏 ขนาดภาพก่อนทำ: {img_pil.size} -> หลังทำ: {final_nobg_pil.size}")
print(f"💾 ขนาดไฟล์ผลลัพธ์: {len(final_bytes) / (1024*1024):.2f} MB")

### 5. แสดงผลลัพธ์สุดท้าย

In [ ]:
def display_with_checkerboard(img, max_width=400):
    from PIL import ImageDraw
    bg = Image.new('RGB', img.size, (255, 255, 255))
    draw = ImageDraw.Draw(bg)
    sq_size = 40
    for y in range(0, img.height, sq_size):
        for x in range(0, img.width, sq_size):
            if (x // sq_size + y // sq_size) % 2 == 0:
                draw.rectangle([x, y, x+sq_size, y+sq_size], fill=(200, 200, 200))
    bg.paste(img, (0, 0), img)
    
    display(bg.resize((max_width, int(max_width * bg.height / bg.width))))

print("✨ ผลลัพธ์สุดท้าย (อัปสเกล 4x -> ลบพื้นหลัง -> 300 DPI)")
display_with_checkerboard(final_nobg_pil)